## Notebook to design random flexible loop sequences of variable lengths

Goal: Design a function with only two arguments - length and seed - to generate flexible loop sequences that satisfy the following criteria:
- The first and last residue must be G. 
- The sequence consists of 50-80% G, the remaining residues should be S.
- There should not be more than 2 S in a row. 
- Then, randomly mutate S into D. The probabily of S > D mutation is between 20% and 50%.
- The sequence generation should be stocastic if seed is None, otherwise use seed to control random number generation. 
- Return the protein sequence as a string. 

In [141]:
import random

def generate_loop_sequence(
    length: int, 
    seed: int = None, 
    s_percent: list = [0.2, 0.4], 
    ds_ratio: list = [0.1, 0.3],
    max_consecutive_g = 5,
    max_consecutive_s = 2
) -> str:
    """
    Generate a flexible loop sequence satisfying specific criteria.
    
    Args:
        length: Length of the sequence (must be >= 1)
        seed: Random seed for reproducibility. If None, generation is stochastic.
        s_percent: List [min, max] for percent of S residues (e.g., [0.2, 0.4]).
        ds_ratio: List [min, max] for D/S mutation ratio (e.g., [0.1, 0.3]).
        max_consecutive_g: Maximum allowed consecutive G residues.
        max_consecutive_s: Maximum allowed consecutive S residues.
    
    Returns:
        A protein sequence string consisting of G, S, and D residues.
        
    Criteria:
        - First and last residue are G
        - S content within s_percent range, remaining are G (before D mutation)
        - No more than max_consecutive_s consecutive S
        - No more than max_consecutive_g consecutive G
        - A controlled number of S residues are randomly selected and mutated to D,
          ensuring the final D/S ratio is between values given in ds_ratio.
    """
    if length < 1:
        raise ValueError("Length must be at least 1")
    
    # Create isolated RNG instance for reproducibility
    rng = random.Random(seed)
    
    # Determine target S count (within provided range, since G should be 50-80%)
    if not (isinstance(s_percent, (list, tuple)) and len(s_percent) == 2):
        raise ValueError("s_percent must be a list or tuple of length 2")
    s_percent_min, s_percent_max = s_percent
    s_percentage = rng.uniform(s_percent_min, s_percent_max)
    target_s_count = round(length * s_percentage)
    
    # Build sequence position by position
    sequence = []
    
    for i in range(length):
        if i == 0 or i == length - 1:
            # First and last must be G
            sequence.append('G')
        else:
            # Count consecutive G before this position
            consecutive_g_before = 0
            for j in range(i - 1, -1, -1):
                if sequence[j] == 'G':
                    consecutive_g_before += 1
                else:
                    break
            
            # Count consecutive S before this position
            consecutive_s_before = 0
            for j in range(i - 1, -1, -1):
                if sequence[j] == 'S':
                    consecutive_s_before += 1
                else:
                    break
            
            # Determine if we MUST place S (to avoid exceeding max_consecutive_g)
            must_place_s = consecutive_g_before >= max_consecutive_g
            
            # Determine if we CAN place S (without exceeding max_consecutive_s)
            can_place_s = consecutive_s_before < max_consecutive_s
            
            # Current S count and remaining positions
            current_s_count = sequence.count('S')
            remaining_positions = length - i - 1  # -1 for the final G
            
            # Calculate how many more S we need/can place
            s_still_needed = target_s_count - current_s_count
            
            if must_place_s and can_place_s:
                sequence.append('S')
            elif must_place_s and not can_place_s:
                # Conflict: can't place S but need to break G run
                # This shouldn't happen with reasonable parameters, place G anyway
                sequence.append('G')
            elif s_still_needed > 0 and can_place_s:
                # Randomly decide based on how many S we still need
                # Higher probability if we need more S relative to remaining positions
                s_probability = s_still_needed / (remaining_positions + 1) if remaining_positions >= 0 else 0
                if rng.random() < s_probability:
                    sequence.append('S')
                else:
                    sequence.append('G')
            else:
                sequence.append('G')
    
    # Mutate a controlled number of S into D to achieve specified D/S ratio range
    s_positions = [i for i, aa in enumerate(sequence) if aa == 'S']
    total_s = len(s_positions)
    
    if total_s > 0:
        # Choose target D/S ratio within specified range
        if not (isinstance(ds_ratio, (list, tuple)) and len(ds_ratio) == 2):
            raise ValueError("ds_ratio must be a list or tuple of length 2")
        d_prob_min, d_prob_max = ds_ratio
        target_d_s_ratio = rng.uniform(d_prob_min, d_prob_max)
        
        # Calculate number of D to create
        # D/S = ratio, and D + S = total_s (original count)
        # So D = total_s * ratio / (1 + ratio)
        num_d = round(total_s * target_d_s_ratio / (1 + target_d_s_ratio))
        num_d = max(0, min(num_d, total_s))  # Ensure valid range
        
        # Randomly select which S positions to mutate to D
        if num_d > 0:
            positions_to_mutate = rng.sample(s_positions, num_d)
            for pos in positions_to_mutate:
                sequence[pos] = 'D'
    
    return ''.join(sequence)


# Function to sanity check the generated sequences
def check_loop_sequence(sequence: str):
    """
    Compute the S content, D/S ratio, and max consecutive G and S residues.
    """
    g_count = sequence.count('G')
    s_count = sequence.count('S')
    d_count = sequence.count('D')
    
    # Count max consecutive G
    consecutive_g = 0
    max_consecutive_g = 0
    for char in sequence:
        if char == 'G':
            consecutive_g += 1
            max_consecutive_g = max(max_consecutive_g, consecutive_g)
        else:
            consecutive_g = 0
    
    # Count max consecutive S
    consecutive_s = 0
    max_consecutive_s = 0
    for char in sequence:
        if char == 'S':
            consecutive_s += 1
            max_consecutive_s = max(max_consecutive_s, consecutive_s)
        else:
            consecutive_s = 0
    
    s_content = (s_count + d_count) / len(sequence)  # S+D since D came from S
    d_s_ratio = d_count / s_count if s_count > 0 else 0
    
    return s_content, d_s_ratio, max_consecutive_g, max_consecutive_s
    


# ------------------------------------------------------------
seq = generate_loop_sequence(
    length = 5,
    s_percent = [0.2, 0.4],
    ds_ratio = [0.1, 0.3],
    max_consecutive_g = 5,
    max_consecutive_s = 2
)

print(seq)

s_content, d_s_ratio, max_consec_g, max_consec_s = check_loop_sequence(seq)
print(f"S+D content: {s_content:.2f}")
print(f"D/S ratio: {d_s_ratio:.2f}")
print(f"Max consecutive G: {max_consec_g}")
print(f"Max consecutive S: {max_consec_s}")



GGSSG
S+D content: 0.40
D/S ratio: 0.00
Max consecutive G: 2
Max consecutive S: 2
